# 🕌 Athar - Upload to Kaggle

**Purpose:** Upload Athar datasets to Kaggle for private team access

**Benefits:**
- ✅ 100 GB free per dataset
- ✅ Private datasets supported
- ✅ Easy Colab integration
- ✅ Good for team collaboration

**Upload Time:** ~1-2 hours

---

## 1️⃣ Install Kaggle CLI

In [ ]:
%%bash
pip install -q kaggle

echo "✅ Kaggle CLI installed"

## 2️⃣ Setup Kaggle API Token

In [ ]:
import os
from google.colab import userdata

print("🔑 Setting up Kaggle API token...")

# Try to get from secrets
try:
    KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
    KAGGLE_KEY = userdata.get('KAGGLE_KEY')
    
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
        os.environ['KAGGLE_KEY'] = KAGGLE_KEY
        print(f"✅ Kaggle username configured: {KAGGLE_USERNAME}")
    else:
        print("⚠️  No Kaggle secrets found")
        print("\n💡 How to get Kaggle API token:")
        print("   1. Go to: https://www.kaggle.com/settings")
        print("   2. Click 'Create New Token'")
        print("   3. Download kaggle.json")
        print("   4. Add to Colab Secrets:")
        print("      - KAGGLE_USERNAME: your username")
        print("      - KAGGLE_KEY: your API key")
        
        # Manual input
        KAGGLE_USERNAME = input("\nEnter Kaggle username: ")
        KAGGLE_KEY = input("Enter Kaggle API key: ")
        
        os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
        os.environ['KAGGLE_KEY'] = KAGGLE_KEY
        print("✅ Token configured")

except Exception as e:
    print(f"❌ Error: {e}")

## 3️⃣ Prepare Dataset Metadata for Kaggle

In [ ]:
import json
from pathlib import Path

# Create Kaggle dataset metadata
dataset_metadata = {
    "title": "Athar Islamic QA System Datasets",
    "id": f"{os.environ.get('KAGGLE_USERNAME', 'username')}/athar-islamic-qa-datasets",
    "licenses": [{"name": "MIT"}],
    "subtitle": "8,425 Islamic books and 650K hadith for QA system",
    "description": """# Athar Islamic QA System - Datasets

Comprehensive Islamic scholarly texts for the Athar QA system.

## Dataset Overview
- **Total Size:** ~19 GB
- **Books:** 8,425 from Shamela Library (17.16 GB)
- **Hadith:** 650,986 from Sanadset (1.43 GB)
- **Language:** Arabic
- **License:** MIT

## Files
- `extracted_books/` - Shamela books in TXT format
- `hadith/sanadset.csv` - Hadith with sanad/matan
- `metadata/` - Categories, authors, books metadata

## Usage
```python
import kaggle

# Download dataset
kaggle.api.dataset_download_files(
    'username/athar-islamic-qa-datasets',
    path='./data',
    unzip=True
)
```

## Citation
```
@misc{athar2026,
  title={Athar Islamic QA System Datasets},
  author={Athar Team},
  year={2026},
  url={https://www.kaggle.com/datasets/username/athar-islamic-qa-datasets}
}
```
""",
    "tags": ["islamic", "quran", "hadith", "arabic", "qa-system", "rag"],
    "isPrivate": True  # Set to False for public dataset
}

# Save metadata
output_dir = Path("/content/drive/MyDrive/Athar/data/athar-datasets")
kaggle_meta = output_dir / "dataset-metadata.json"
with open(kaggle_meta, 'w') as f:
    json.dump(dataset_metadata, f, indent=2)

print(f"✅ Created Kaggle metadata: {kaggle_meta}")
print(f"\n📋 Dataset ID: {dataset_metadata['id']}")
print(f"   Private: {dataset_metadata['isPrivate']}")

## 4️⃣ Upload to Kaggle

In [ ]:
import kaggle
from pathlib import Path

DATASET_DIR = "/content/drive/MyDrive/Athar/data/athar-datasets"
DATASET_ID = dataset_metadata['id']

print(f"📤 Uploading dataset to Kaggle...")
print(f"   Dataset ID: {DATASET_ID}")
print(f"   Directory: {DATASET_DIR}")
print(f"\n⏱️  Estimated time: 1-2 hours")

try:
    # Create new dataset
    kaggle.api.dataset_create_new(
        folder_path=DATASET_DIR,
        public=not dataset_metadata['isPrivate']
    )
    print(f"\n✅ Dataset created successfully!")
except Exception as e:
    print(f"\n⚠️  Dataset might already exist, updating instead...")
    print(f"   Error: {e}")
    
    try:
        # Update existing dataset
        kaggle.api.dataset_metadata_update(
            dataset=DATASET_ID,
            metadata_body=dataset_metadata
        )
        
        # Upload files
        for f in Path(DATASET_DIR).rglob('*'):
            if f.is_file():
                rel_path = f.relative_to(Path(DATASET_DIR))
                print(f"   📤 Uploading: {rel_path}")
                kaggle.api.dataset_upload_file(
                    dataset=DATASET_ID,
                    file_name=str(f)
                )
        
        print(f"\n✅ Dataset updated successfully!")
    except Exception as e2:
        print(f"\n❌ Error updating: {e2}")

print(f"\n🔗 View at: https://www.kaggle.com/datasets/{DATASET_ID}")

## 5️⃣ Test Download

In [ ]:
import kaggle
import os

print("📥 Testing download...")

# Download to test directory
test_dir = "/content/test_download"
os.makedirs(test_dir, exist_ok=True)

kaggle.api.dataset_download_files(
    DATASET_ID,
    path=test_dir,
    unzip=False
)

# List downloaded files
print(f"\n✅ Downloaded files:")
for f in Path(test_dir).rglob('*'):
    if f.is_file():
        size = f.stat().st_size / 1e9
        print(f"   {f.name:40s} {size:.2f} GB")

print(f"\n🎉 Kaggle dataset is ready!")